# Model A — Cross-Entropy baseline

5-fold GroupKFold, no augmentation. Kaggle public LB: 0.773.


In [1]:
from google.colab import drive

import os
import sys
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/BirdCLEF_Project")
REPRO_ROOT = DRIVE_PROJECT / "repro"

if not (REPRO_ROOT / "birdclef").exists():
    raise FileNotFoundError(
        f"Expected repro at {REPRO_ROOT}. Place repro inside BirdCLEF_Project on Drive."
    )

sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)
print(f"Working directory: {REPRO_ROOT}")

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn

from birdclef.colab import colab_prepare_training

repro_root = colab_prepare_training(stage_tta=False)
print(f"Project root: {repro_root}")


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.9 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df, NUM_CLASSES, _ = prepare_baseline_data()
print(f"Samples: {len(df)}, classes: {NUM_CLASSES}")
print(df["fold"].value_counts().sort_index())

Samples: 35549, classes: 234
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64


In [4]:
SAVE_DIR = MODELS_DIR / "ce_baseline_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 10
baseline_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0

    # Split data by fold
    train_df = df[df["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df[df["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=False)
    valid_ds = BirdDataset(valid_df, is_tta=False)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    # Fresh model/optimizer per fold so each fold is an independent CV estimate.
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)

        # One-hot encode labels for AUC computation
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    baseline_fold_scores.append(best_auc)

print(f"CV score: {np.mean(baseline_fold_scores):.4f} (+/- {np.std(baseline_fold_scores):.4f})")

100%|██████████| 7110/7110 [00:01<00:00, 3847.88it/s]


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2007 | Val Loss: 0.8381 | Val AUC: 0.9908
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.4874 | Val Loss: 0.8266 | Val AUC: 0.9905
Epoch 03/10 | Train Loss: 0.2842 | Val Loss: 0.8745 | Val AUC: 0.9903
Epoch 04/10 | Train Loss: 0.1967 | Val Loss: 0.9122 | Val AUC: 0.9876
Epoch 05/10 | Train Loss: 0.1553 | Val Loss: 0.9373 | Val AUC: 0.9884
Epoch 06/10 | Train Loss: 0.1423 | Val Loss: 0.9535 | Val AUC: 0.9883
Epoch 07/10 | Train Loss: 0.1436 | Val Loss: 0.9473 | Val AUC: 0.9892
Epoch 08/10 | Train Loss: 0.1350 | Val Loss: 0.9654 | Val AUC: 0.9870
Epoch 09/10 | Train Loss: 0.1235 | Val Loss: 0.9735 | Val AUC: 0.9863
Epoch 10/10 | Train Loss: 0.1288 | Val Loss: 1.0213 | Val AUC: 0.9847
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/learning_curves

100%|██████████| 7110/7110 [00:01<00:00, 4012.34it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2003 | Val Loss: 0.8427 | Val AUC: 0.9913
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.4914 | Val Loss: 0.8258 | Val AUC: 0.9914
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold1.pth
Epoch 03/10 | Train Loss: 0.2826 | Val Loss: 0.8763 | Val AUC: 0.9918
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold1.pth
Epoch 04/10 | Train Loss: 0.1954 | Val Loss: 0.8906 | Val AUC: 0.9913
Epoch 05/10 | Train Loss: 0.1546 | Val Loss: 0.9185 | Val AUC: 0.9915
Epoch 06/10 | Train Loss: 0.1448 | Val Loss: 0.9344 | Val AUC: 0.9904
Epoch 07/10 | Train Loss: 0.1455 | Val Loss: 0.9541 | Val AUC: 0.9890
Epoch 08/10 | Train Loss: 0.1372 | Val Loss: 0.9804 | Val AUC: 0.9888
Epoch 09/10 | Train Loss: 0.1328 |

100%|██████████| 7110/7110 [00:01<00:00, 4628.20it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2237 | Val Loss: 0.7855 | Val AUC: 0.9903
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.4941 | Val Loss: 0.7776 | Val AUC: 0.9907
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold2.pth
Epoch 03/10 | Train Loss: 0.2909 | Val Loss: 0.8014 | Val AUC: 0.9893
Epoch 04/10 | Train Loss: 0.1984 | Val Loss: 0.8405 | Val AUC: 0.9890
Epoch 05/10 | Train Loss: 0.1669 | Val Loss: 0.8543 | Val AUC: 0.9889
Epoch 06/10 | Train Loss: 0.1436 | Val Loss: 0.8967 | Val AUC: 0.9886
Epoch 07/10 | Train Loss: 0.1451 | Val Loss: 0.8932 | Val AUC: 0.9890
Epoch 08/10 | Train Loss: 0.1386 | Val Loss: 0.9052 | Val AUC: 0.9882
Epoch 09/10 | Train Loss: 0.1241 | Val Loss: 0.9136 | Val AUC: 0.9887
Epoch 10/10 | Train Loss: 0.1349 | Val Loss: 0.9138 | Val AUC: 0.9885
Learni

100%|██████████| 7110/7110 [00:01<00:00, 4601.92it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 1.2201 | Val Loss: 0.8081 | Val AUC: 0.9895
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.5022 | Val Loss: 0.7806 | Val AUC: 0.9908
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold3.pth
Epoch 03/10 | Train Loss: 0.2981 | Val Loss: 0.8142 | Val AUC: 0.9913
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold3.pth
Epoch 04/10 | Train Loss: 0.2090 | Val Loss: 0.8125 | Val AUC: 0.9911
Epoch 05/10 | Train Loss: 0.1605 | Val Loss: 0.8365 | Val AUC: 0.9898
Epoch 06/10 | Train Loss: 0.1462 | Val Loss: 0.8576 | Val AUC: 0.9895
Epoch 07/10 | Train Loss: 0.1456 | Val Loss: 0.8912 | Val AUC: 0.9875
Epoch 08/10 | Train Loss: 0.1380 | Val Loss: 0.8919 | Val AUC: 0.9879
Epoch 09/10 | Train Loss: 0.1304 |

100%|██████████| 7109/7109 [00:01<00:00, 4115.22it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 1.2203 | Val Loss: 0.7817 | Val AUC: 0.9907
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.5004 | Val Loss: 0.7778 | Val AUC: 0.9929
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/ce_baseline_10ep/best_moe_fold4.pth
Epoch 03/10 | Train Loss: 0.3005 | Val Loss: 0.8000 | Val AUC: 0.9910
Epoch 04/10 | Train Loss: 0.1969 | Val Loss: 0.8203 | Val AUC: 0.9906
Epoch 05/10 | Train Loss: 0.1585 | Val Loss: 0.8447 | Val AUC: 0.9903
Epoch 06/10 | Train Loss: 0.1560 | Val Loss: 0.8730 | Val AUC: 0.9895
Epoch 07/10 | Train Loss: 0.1392 | Val Loss: 0.9030 | Val AUC: 0.9881
Epoch 08/10 | Train Loss: 0.1446 | Val Loss: 0.8862 | Val AUC: 0.9893
Epoch 09/10 | Train Loss: 0.1236 | Val Loss: 0.9132 | Val AUC: 0.9874
Epoch 10/10 | Train Loss: 0.1365 | Val Loss: 0.9166 | Val AUC: 0.9889
Learni